In [10]:
import os
import pandas as pd
import numpy as np
path = os.path.join( "..", "data", "processed", "accidents_clean.csv")
df = pd.read_csv(path)

df.drop(columns=['catv_regroup','equipements','age','col','trajet','age',
                 'sexe','catu','place','motor','choc','manv','jour','mois','an','hrmn'], inplace=True)

In [13]:
#################################################################
#  ACCIDENTS + tags routiers (valeur ≤30 m) + dist hôpital/pompier
#                 France métropolitaine  •  v2025‑07
#################################################################
import time, warnings, requests, json
from pathlib import Path
import pandas as pd, geopandas as gpd, numpy as np
from shapely.geometry import Point, shape
import osmnx as ox                     # seulement pour bbox helpers

warnings.filterwarnings("ignore", category=UserWarning)

# --------------------- CONFIG ---------------------------------
OUT_FILE  = "../data/accidents_enrichis.parquet"
ROUTE_DIR = Path("./routes"); ROUTE_DIR.mkdir(exist_ok=True)

TAGS        = ["bridge", "tunnel", "lit", "traffic_calming"]
OVERPASS    = "https://overpass.kumi.systems/api/interpreter"
MAX_METERS  = 30           # rayon tags routiers (mètres)   ←★

TILES = [(lat, lon) for lon in range(-5, 10) for lat in range(41, 52)]
SLEEP = 3                  # pause entre requêtes

# --------------------- 1. Accident GeoDataFrame --------------
assert {"lat", "long"}.issubset(df.columns)

mask_fr = df.eval("41<=lat<=51 and -5<=long<=9")
df_fr = df[mask_fr].copy().reset_index(drop=True)

gdf_pts = gpd.GeoDataFrame(
    df_fr, geometry=gpd.points_from_xy(df_fr.long, df_fr.lat), crs="EPSG:4326")
print(f"✅ accidents retenus : {len(gdf_pts):,}")

# --------------------- 2. Téléchargement routes taggées -------
def download_routes_tile(lat, lon):
    fn = ROUTE_DIR / f"routes_{lat}_{lon}.parquet"
    if fn.exists() and fn.stat().st_size:
        print(f"⏩ {fn.name} déjà présent")
        return
    bbox = f"{lat},{lon},{lat+1},{lon+1}"
    query = "[out:json][timeout:40];(" + "".join(
        [f'way["highway"]["{t}"]({bbox});' for t in TAGS]
    ) + ");out geom;"
    try:
        r = requests.get(OVERPASS, params={"data": query}, timeout=90)
        r.raise_for_status()
        feats = []
        for w in r.json()["elements"]:
            coords=[(n["lon"],n["lat"]) for n in w.get("geometry",[])]
            if len(coords)>=2:
                tags=w.get("tags",{})
                feats.append({t:tags.get(t) for t in TAGS}|
                             {"geometry":shape({"type":"LineString","coordinates":coords})})
        if feats:
            gpd.GeoDataFrame(feats, crs="EPSG:4326").to_parquet(fn)
            print(f"✓ {fn.name:<18} {len(feats):,}")
        else:
            print(f"⏭ tuile {lat},{lon} vide")
    except Exception as e:
        print(f"❌ routes {lat},{lon} : {e}")
    finally:
        time.sleep(SLEEP)

print("\n⏳ Vérif / download routes…")
for lat, lon in TILES:
    download_routes_tile(lat, lon)

# Lecture routes valides
frames=[]
for p in ROUTE_DIR.glob("routes_*.parquet"):
    try:
        dfp=pd.read_parquet(p)
        if "geometry" in dfp and dfp.geometry.notna().any():
            if isinstance(dfp.geometry.iloc[0], bytes):
                dfp["geometry"]=gpd.GeoSeries.from_wkb(dfp.geometry)
            frames.append(dfp)
    except Exception as e:
        print(f"⚠️ {p.name} ignoré ({e})")
if not frames:
    raise RuntimeError("❌ Aucun tronçon taggé valide")
gdf_routes = gpd.GeoDataFrame(pd.concat(frames, ignore_index=True),
                              geometry="geometry", crs="EPSG:4326")
print(f"🛣️ tronçons taggés : {len(gdf_routes):,}")

# --------------------- 3. Fonctions POI Overpass --------------
def fetch_pois(bbox, filters):
    query = f"[out:json][timeout:25];(" + "".join(
        [f'{obj}{flt}({bbox});' for obj in ["node","way","relation"] for flt in filters]
    ) + ");out center;"
    r = requests.get(OVERPASS, params={"data": query}, timeout=60)
    r.raise_for_status()
    feats=[]
    for el in r.json()["elements"]:
        if "lon" in el and "lat" in el:
            feats.append({"geometry": Point(el["lon"], el["lat"])})
        elif "center" in el:
            c=el["center"]; feats.append({"geometry":Point(c["lon"],c["lat"])})
    return feats

def collect_pois(tag_query):
    parts=[]
    for lat, lon in TILES:
        bbox=f"{lat},{lon},{lat+1},{lon+1}"
        try:
            feats=fetch_pois(bbox, tag_query)
            if feats: parts.extend(feats)
        except Exception as e:
            print(f"⚠️ POI {lat},{lon} : {e}")
        time.sleep(1)
    if not parts: return gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")
    return gpd.GeoDataFrame(parts, crs="EPSG:4326")

print("\n⏳ Téléchargement hôpitaux…")
gdf_hop  = collect_pois(['["amenity"="hospital"]'])
print("⏳ Téléchargement casernes…")
gdf_fire = collect_pois(['["amenity"="fire_station"]'])
print(f"🏥 {len(gdf_hop):,} hôpitaux | 🚒 {len(gdf_fire):,} casernes")

# --------------------- 4. Jointures métriques -----------------
proj="EPSG:3857"
pts_m    = gdf_pts.to_crs(proj).reset_index(drop=True)
routes_m = gdf_routes.to_crs(proj).reset_index(drop=True)
hop_m    = gdf_hop.to_crs(proj).reset_index(drop=True)
fire_m   = gdf_fire.to_crs(proj).reset_index(drop=True)

# tags routiers à ≤ MAX_METERS
for tag in TAGS:
    sub = routes_m[routes_m[tag].notna()][["geometry", tag]]
    join = gpd.sjoin_nearest(pts_m, sub, how="left",
                             distance_col="dist", max_distance=MAX_METERS)
    col = f"{tag}_right" if f"{tag}_right" in join.columns else tag
    pts_m[tag] = join[col].reset_index(drop=True)

# distances hôpital / caserne (sans limite)
def dist_nearest(src, tgt):
    if tgt.empty: return np.nan
    j = gpd.sjoin_nearest(src, tgt, how="left", distance_col="d")
    return j["d"].reset_index(drop=True)
pts_m["dist_hop_m"]  = dist_nearest(pts_m, hop_m)
pts_m["dist_fire_m"] = dist_nearest(pts_m, fire_m)

# --------------------- 5. Fusion & export ----------------------
enrich = pts_m.to_crs("EPSG:4326").drop(columns="geometry")
df_final = df_fr.join(
    enrich[TAGS+["dist_hop_m","dist_fire_m"]].set_index(df_fr.index)
)

df_final.to_parquet(OUT_FILE, index=False)
print(f"\n🎉 Fichier enrichi ➜ {OUT_FILE}")
print(df_final.head(3))


✅ accidents retenus : 580,067

⏳ Vérif / download routes…
⏩ routes_41_-5.parquet déjà présent
⏩ routes_42_-5.parquet déjà présent
⏩ routes_43_-5.parquet déjà présent
⏭ tuile 44,-5 vide
⏭ tuile 45,-5 vide
⏭ tuile 46,-5 vide
⏩ routes_47_-5.parquet déjà présent
⏩ routes_48_-5.parquet déjà présent
⏭ tuile 49,-5 vide
⏩ routes_50_-5.parquet déjà présent
⏩ routes_51_-5.parquet déjà présent
⏩ routes_41_-4.parquet déjà présent
⏩ routes_42_-4.parquet déjà présent
⏩ routes_43_-4.parquet déjà présent
⏭ tuile 44,-4 vide
⏭ tuile 45,-4 vide
⏭ tuile 46,-4 vide
⏩ routes_47_-4.parquet déjà présent
⏩ routes_48_-4.parquet déjà présent
⏭ tuile 49,-4 vide
⏩ routes_50_-4.parquet déjà présent
⏩ routes_51_-4.parquet déjà présent
⏩ routes_41_-3.parquet déjà présent
⏩ routes_42_-3.parquet déjà présent
⏩ routes_43_-3.parquet déjà présent
⏭ tuile 44,-3 vide
⏭ tuile 45,-3 vide
⏩ routes_46_-3.parquet déjà présent
⏩ routes_47_-3.parquet déjà présent
⏩ routes_48_-3.parquet déjà présent
⏩ routes_49_-3.parquet déjà prés

In [14]:
# ------------------------------------------------------------------
# TMJA
# ------------------------------------------------------------------
import geopandas as gpd
from shapely.geometry import Point

# 1)  GeoDataFrame des accidents (géométrie recréée)
gdf_acc = gpd.GeoDataFrame(
    df_final,                                  # <-- ton DataFrame existant
    geometry=[Point(xy) for xy in zip(df_final["long"], df_final["lat"])],
    crs="EPSG:4326"
)

# 2)  Lecture de la couche TMJA (adapte le chemin et le nom de champ)
tmja_path = "../data/raw/tmja2019-shp/TMJA2019.shp"                   # ou .geojson, .gpkg, …
gdf_tmja = (
    gpd.read_file(tmja_path)
      .set_crs("EPSG:2154")           # ← attribution manuelle du CRS
      .to_crs("EPSG:4326")            # ← reprojection vers WGS84
      .loc[:, ["TMJA", "geometry"]]   # ← garde les colonnes utiles
)

print("✅ compteurs TMJA chargés :", len(gdf_tmja))

# 3)  Jointure spatiale : compteur le plus proche (≤ 50 m)
max_dist_deg = 50 / 111_000                   # 50 m → ~0,00045°
gdf_join = gpd.sjoin_nearest(
    gdf_acc,
    gdf_tmja,
    how="left",
    distance_col="dist_tmja",
    max_distance=max_dist_deg
).drop(columns="index_right")

# 4)  Résultat sans géométrie
df_final_tmja = gdf_join.drop(columns="geometry")

print("🎉  Aperçu avec TMJA ajouté :")
print(df_final_tmja[["TMJA", "dist_tmja"]].describe())

# (option) enregistrer
# df_final_tmja.to_parquet("accidents_enrichis_tmja.parquet")

✅ compteurs TMJA chargés : 4695
🎉  Aperçu avec TMJA ajouté :
                TMJA     dist_tmja
count  102152.000000  1.021520e+05
mean    59246.962184  1.086303e-04
std     57854.742417  1.038282e-04
min         0.000000  1.480489e-11
25%         0.000000  1.803380e-05
50%     45572.000000  8.159275e-05
75%    100582.000000  1.734417e-04
max    238387.000000  4.504199e-04


In [16]:
df_final_tmja = df_final_tmja.drop(columns=["dist_tmja"])

In [ ]:
df_final_tmja['TMJA'] = df_final_tmja['TMJA'].fillna(0)


In [34]:
df_final_tmja['lit'] = df_final_tmja['lit'].where(
    df_final_tmja['lit'] == 'no',    # condition : on garde ‘no’
    'yes'                                # sinon on met ‘yes’
)

df_final_tmja['tunnel'] = df_final_tmja['tunnel'].where(
    df_final_tmja['tunnel'] == 'no',    # condition : on garde ‘no’
    'yes'                                # sinon on met ‘yes’
)

df_final_tmja['bridge'] = df_final_tmja['bridge'].where(
    df_final_tmja['bridge'] == 'no',    # condition : on garde ‘no’
    'yes'                                # sinon on met ‘yes’
)



df_final_tmja['traffic_calming'] = df_final_tmja['traffic_calming'].where(
    df_final_tmja['traffic_calming'] == 'no',    # condition : on garde ‘no’
    'yes'                                # sinon on met ‘yes’
)

In [36]:
# Remplacement des zéros dans TMJA par des valeurs cohérentes basés sur la hiérarchie d’imputation des routes aux caractéristiques similaires

import numpy as np
import pandas as pd

df = df_final_tmja.copy()

# 1) Zéros -> NaN
df['TMJA'] = df['TMJA'].replace(0, np.nan)

# 2) Hiérarchie d’imputation
hierarchy = [
    ['catr','com','agg'],
    ['catr','com'],
    ['catr','dep','agg'],
    ['catr','dep'],
    ['catr']
]

for cols in hierarchy:
    df['TMJA'] = df['TMJA'].fillna(
        df.groupby(cols, dropna=False)['TMJA'].transform('median')
    )



In [37]:
df.to_csv("../data/processed/accidents_geo_enriched.csv", index=False)

In [30]:
df.head(30)

,obs,obsm,grav,lum,dep,com,agg,int,atm,lat,...,infra,situ,vma,bridge,tunnel,lit,traffic_calming,dist_hop_m,dist_fire_m,TMJA
0,0.0,2.0,2,4,93,93053,1,1.0,1.0,48.896210,...,2.0,1.0,70,viaduct,no,yes,no,3022.616104,2602.367034,118500.0
1,0.0,2.0,2,4,93,93053,1,1.0,1.0,48.896210,...,2.0,1.0,70,viaduct,no,yes,no,3022.616104,2602.367034,118500.0
2,1.0,0.0,1,4,93,93053,1,1.0,1.0,48.896210,...,2.0,1.0,70,viaduct,no,yes,no,3022.616104,2602.367034,118500.0
3,4.0,0.0,2,3,93,93066,1,1.0,1.0,48.930700,...,0.0,1.0,70,no,no,yes,no,799.172524,2175.243926,132186.0
4,0.0,2.0,1,1,92,92036,1,1.0,1.0,48.935872,...,0.0,1.0,90,no,no,yes,no,2509.148777,2255.222666,92091.0
5,0.0,2.0,2,1,92,92036,1,1.0,1.0,48.935872,...,0.0,1.0,90,no,no,yes,no,2509.148777,2255.222666,92091.0
6,1.0,0.0,2,1,92,92036,1,1.0,1.0,48.935872,...,0.0,1.0,90,no,no,yes,no,2509.148777,2255.222666,92091.0
7,0.0,2.0,1,1,92,92036,1,1.0,1.0,48.935872,...,0.0,1.0,90,no,no,yes,no,2509.148777,2255.222666,92091.0
8,0.0,2.0,1,5,94,94069,1,1.0,1.0,48.817329,...,0.0,1.0,90,no,no,yes,no,584.907213,2892.535565,221626.0
9,0.0,2.0,1,5,94,94069,1,1.0,1.0,48.817329,...,0.0,1.0,90,no,no,yes,no,584.907213,2892.535565,221626.0
